In [1]:
import gymnasium as gym
from minigrid.core.constants import OBJECT_TO_IDX

In [2]:
env = gym.make("MiniGrid-BlockedUnlockPickup-v0")

In [9]:
print(dir(env.unwrapped))

['__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_gen_grid', '_gen_mission', '_is_protocol', '_np_random', '_np_random_seed', '_rand_bool', '_rand_color', '_rand_elem', '_rand_float', '_rand_int', '_rand_pos', '_rand_subset', '_reward', 'action_space', 'actions', 'add_distractors', 'add_door', 'add_object', 'agent_dir', 'agent_pos', 'agent_pov', 'agent_sees', 'agent_view_size', 'carrying', 'clock', 'close', 'connect_all', 'dir_vec', 'front_pos', 'gen_obs', 'gen_obs_grid', 'get_frame', 'get_full_render', 'get_pov_render', 'get_room', 'get_view_coords', 'get_view_exts', 'get_wrapper_a

In [17]:
class CustomRewardWrapper(gym.RewardWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.last_carrying = None
        self.door_was_locked = True
    def reward(self, reward):
        # 1. 基础：原有的成功奖励 (通常是任务完成时的奖励)
        new_reward = reward
        
        # 获取未包装的原始环境引用以读取内部状态
        unwrapped = self.env.unwrapped
        carrying = unwrapped.carrying
        
        # 2. 塑形奖励 A：捡起钥匙奖励
        # 如果之前没拿东西，现在拿到了钥匙 (OBJECT_TO_IDX['key'] = 5)
        if self.last_carrying is None and carrying is not None:
            if carrying.type == 'key':
                new_reward += 0.2  # 捡起钥匙给 0.2
        
        # 3. 塑形奖励 B：成功开门奖励
        # 检查环境中的门是否从锁住状态变成了开启或关闭状态
        # 这种检查比较底层，通常需要遍历 grid 或检查特定位置
        # 简化版逻辑：
        if self.door_was_locked:
            # 这里可以根据你的逻辑判断门是否开了，比如检查当前格子的动作
            # 或者判断门的位置状态是否改变
            pass

        # 更新状态追踪
        self.last_carrying = carrying
        
        # 4. 惩罚：每走一步扣除微小分数，鼓励最短路径
        new_reward -= 0.001
        
        return new_reward